# Reading data
Read the train.csv file as a pandas dataframe.

In [33]:
import pandas as pd
titanic = pd.read_csv("datatitanic/train.csv")
titanic = titanic.set_index("PassengerId")
titanic.iloc[:10]
titanic

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S


# Indexing
1. Create a function that returns the name of a passenger given their PassengerId.
2. Create a function that returns the PassengerId of a passenger given their Name.
3. Print a message with the ID of passenger **Montvila, Rev. Juozas** with the following format: 'The ID pf passenger Montvila, Rev. Juozas is ##'
4. Print a message with the name of the passenger with ID **42** with the following format: 'The passenger with ID 42 is X'

5. Print all information about the oldest passenger.

In [35]:
# punto 1: Importar funciones y cargar datos

import pandas as pd
import os
from titanic_fns import get_name, get_id, survivors_over_60, survival_percentage
# Asegurar que exista la carpeta 'data'
if not os.path.exists("data"):
    os.makedirs("data")

# Cargar dataset
titanic_df = pd.read_csv("./datatitanic/train.csv")

In [3]:
# Punto 3
montvila_id = get_id(titanic_df, "Montvila, Rev. Juozas")
print(f"[Punto 3] ID de Montvila, Rev. Juozas: {montvila_id}")

# Punto 4
passenger_42_name = get_name(titanic_df, 42)

print(f"[Punto 4] Pasajero con ID 42: {passenger_42_name}")

[Punto 3] ID de Montvila, Rev. Juozas: 887
[Punto 4] Pasajero con ID 42: Turpin, Mrs. William John Robert (Dorothy Ann Wonnacott)


In [25]:
# Punto 5: Información del pasajero más viejo
df_age_clean = titanic_df.dropna(subset=['Age'])
if not df_age_clean.empty:
    oldest_age = df_age_clean['Age'].max()
    oldest_passengers = df_age_clean[df_age_clean['Age'] == oldest_age]
    print("Información del pasajero más viejo:")
    print(oldest_passengers)
else:
    print("No hay datos de edad disponibles.")

Información del pasajero más viejo:
     PassengerId  Survived  Pclass                                  Name  \
630          631         1       1  Barkworth, Mr. Algernon Henry Wilson   

      Sex   Age  SibSp  Parch Ticket  Fare Cabin Embarked Category  
630  male  80.0      0      0  27042  30.0   A23        S      Man  


# Subseting
We are asked to share data for analysis by a third party. Since our dataset contains personal details, we only want to share with them the following information: ticket classes, fares and port of embarkation. We are asked to deliver a sample of the first 100 rows of this dataset.

6. Create and save the new dataset in **data/port_fares.csv**.

In [29]:
# Punto 6: Subconjunto (clase, tarifa, puerto)
subset = titanic_df[["Pclass", "Fare", "Embarked"]].head(100)
subset.to_csv("data/port_fares.csv", index=False)
print("[Punto 6] Subconjunto guardado en data/port_fares.csv")

[Punto 6] Subconjunto guardado en data/port_fares.csv


# Counting
7. We want to know if there were any survivors over the age of 60, print all of their information.
8. How many people over 60 survived?
9. What percentage of people over 60 survived?

In [37]:
# Puntos 7-9: Análisis de sobrevivientes
survivors = survivors_over_60(titanic_df)
count = len(survivors)
percentage = survival_percentage(titanic_df)

print("\n[Punto 7] Sobrevivientes >60 años:")
print(survivors[["PassengerId", "Name", "Age", "Survived"]])
print(f"\n[Punto 8] Total: {count}")
print(f"[Punto 9] Porcentaje: {percentage:.2f}%")


[Punto 7] Sobrevivientes >60 años:
     PassengerId                                       Name   Age  Survived
275          276          Andrews, Miss. Kornelia Theodosia  63.0         1
483          484                     Turkula, Mrs. (Hedwig)  63.0         1
570          571                         Harris, Mr. George  62.0         1
630          631       Barkworth, Mr. Algernon Henry Wilson  80.0         1
829          830  Stone, Mrs. George Nelson (Martha Evelyn)  62.0         1

[Punto 8] Total: 5
[Punto 9] Porcentaje: 22.73%


# Women and children first?
10. Find out if women and children were more likely to survive.

In [16]:
# Question 10: Women and children survival analysis
titanic_df['Category'] = titanic_df.apply(lambda row: 'Woman' if row['Sex'] == 'female' else ('Child' if row['Age'] < 18 else 'Man'), axis=1)
survival_rates = titanic_df.groupby('Category')['Survived'].mean() * 100
print("Survival Rates by Category:")
print(survival_rates)

Survival Rates by Category:
Category
Child    39.655172
Man      16.570328
Woman    74.203822
Name: Survived, dtype: float64


11. Write a function that returns the percentage of people that survived from a subset given as a boolean Pandas series.

In [50]:
# Punto 11: Función para porcentaje de supervivencia
def porcentaje_sobrevivencia(subconjunto: pd.Series) -> float:
    if not isinstance(subconjunto, pd.Series) or subconjunto.dtype != bool:
        raise ValueError("El subconjunto debe ser una serie booleana de Pandas.")
    total = subconjunto.sum()
    if total == 0:
        return 0.0 
    sobrevivientes = titanic_df.loc[subconjunto, 'Survived'].sum()
    return (sobrevivientes / total) * 100

# Summarizing

12. What is the median age of the passengers?
13. How many passengers embarked from each port?

In [36]:
# Punto 12: Mediana de edad
mediana_edad = titanic_df['Age'].median()
print(f"[Punto 12]La mediana de edad de los pasajeros es {mediana_edad:.1f} años.")
# Punto 13: Pasajeros por puerto de embarque
conteo_embarque = titanic_df['Embarked'].value_counts(dropna=False)
print("[Punto 13] Número de pasajeros por puerto de embarque:")
print(conteo_embarque)

[Punto 12]La mediana de edad de los pasajeros es 28.0 años.
[Punto 13] Número de pasajeros por puerto de embarque:
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


14. Generate two hypotheses about how does the survival rate differ among groups of passengers. Write your code to explore both hypotheses.

In [32]:
# Punto 14: Hipótesis
# Hipótesis 1: Supervivencia por clase
sobrevivencia_clase = titanic_df.groupby('Pclass')['Survived'].mean() * 100
print("Tasa de supervivencia por clase:")
print(sobrevivencia_clase)

# Hipótesis 2: Supervivencia por sexo
sobrevivencia_sexo = titanic_df.groupby('Sex')['Survived'].mean() * 100
print("\nTasa de supervivencia por sexo:")
print(sobrevivencia_sexo)

Tasa de supervivencia por clase:
Pclass
1    62.962963
2    47.282609
3    24.236253
Name: Survived, dtype: float64

Tasa de supervivencia por sexo:
Sex
female    74.203822
male      18.890815
Name: Survived, dtype: float64
